# Notebook 05: Retrieval Evaluation

## Overview

This notebook demonstrates how retrieval quality is measured:

```
Test Questions (66)  -->  Batch Retrieve  -->  Score Relevance  -->  Metrics
```

**What you will see:**
- The bilingual test question set
- How relevance is determined (recommendation numbers + page matching)
- How MAP@k, MRR, and found rate are calculated
- Per-language performance breakdown
- Experiment comparison (top-k, chunk size, embedding model)

**Source files:**
- `src/rag_app/evaluation/evaluator.py` - Batch evaluation orchestration
- `src/rag_app/evaluation/metrics.py` - Relevance matching and scoring

## 1. Setup

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.rag_app import config
from src.rag_app.evaluation.evaluator import load_questions, evaluate, build_summary, print_summary
from src.rag_app.evaluation.metrics import (
    expected_recommendations,
    contains_recommendation,
    page_numbers,
    is_relevant,
    average_precision_at_k,
)

## 2. The Test Question Set

66 bilingual questions (33 Arabic + 33 English), each mapped to expected NICE recommendations.
2 additional out-of-scope questions test the refusal behavior.

In [ ]:
questions = load_questions()
print(f"Total questions: {len(questions)}")
print(f"\nSample questions:")
for q in questions[:6]:
    print(f"  [{q['language']}] {q['text'][:60]}...")
    print(f"    Expected: {q['expected_source'][:60]}")
    print()

## 3. Relevance Matching

A retrieved chunk is **relevant** if it contains the expected recommendation number, or if it's about Table 1 on pages 10-12.

In [ ]:
# Example: expanding recommendation ranges
source = "Recommendations 1.5.14-1.5.17"
expanded = expected_recommendations(source)
print(f"Source: '{source}'")
print(f"Expanded: {expanded}")

# Example: checking if a document contains a recommendation
doc_text = "- 1.5.14 Offer oxaliplatin-based combination chemotherapy..."
print(f"\nDocument contains 1.5.14: {contains_recommendation(doc_text, '1.5.14')}")
print(f"Document contains 1.5.15: {contains_recommendation(doc_text, '1.5.15')}")

# Example: page number expansion
print(f"\nPages '10-12': {page_numbers('10-12')}")
print(f"Pages '15': {page_numbers('15')}")

## 4. Run Evaluation

In [ ]:
top_k = 5
report = evaluate(top_k)
summary = build_summary(report)

print_summary(report, summary)

## 5. Metrics Explained

| Metric | What it measures | Formula |
|--------|-----------------|--------|
| **Found rate** | % of questions where expected evidence is in top-k | found / total scored |
| **Precision@k** | % of retrieved chunks that are relevant | relevant_in_top_k / k |
| **MAP@k** | Average precision across all queries | mean(AP@k) |
| **MRR** | How early the first relevant result appears | mean(1/rank_of_first) |

In [ ]:
# Show detailed metrics
print(f"Evaluation Metrics (top-k={summary['top_k']}):")
print(f"  Total questions: {summary['total_questions']}")
print(f"  Scored questions: {summary['scored_questions']}")
print(f"  Out-of-scope (refusal): {summary['out_of_scope_questions']}")
print(f"  Found expected evidence: {summary['found_expected_evidence']}")
print(f"  Found rate: {summary['found_rate']:.1%}")
print(f"  Mean Precision@{top_k}: {summary['mean_precision_at_k']:.4f}")
print(f"  MAP@{top_k}: {summary['map_at_k']:.4f}")
print(f"  MRR: {summary['mrr']:.4f}")

# Per-language breakdown
for lang in ['ar', 'en']:
    key = f"{lang}_found_rate"
    if key in summary:
        print(f"\n  {lang.upper()} found rate: {summary[f'{lang}_found_count']} ({summary[key]:.1%})")

## 6. Failed Questions Analysis

Understanding which questions fail helps identify retrieval gaps.

In [ ]:
failed = [row for row in report if row["status"] == "FAIL"]
print(f"Failed questions: {len(failed)}")
for row in failed:
    print(f"\n  [{row['language']}] {row['question'][:60]}...")
    print(f"    Expected: {row['expected_source'][:50]}")
    print(f"    Top score: {row['top_score']}, Top chunk: {row['top_chunk_id']}")

## 7. Experiment Comparison: Top-K

How does changing k affect found rate vs precision?

In [ ]:
k_values = [1, 3, 5, 10]
experiment_results = []

for k in k_values:
    exp_report = evaluate(k)
    exp_summary = build_summary(exp_report)
    experiment_results.append({
        "k": k,
        "found_rate": exp_summary["found_rate"],
        "precision": exp_summary["mean_precision_at_k"],
        "map": exp_summary["map_at_k"],
        "mrr": exp_summary["mrr"],
    })
    print(f"k={k:>2}: found={exp_summary['found_rate']:.1%}  "
          f"precision={exp_summary['mean_precision_at_k']:.4f}  "
          f"MAP={exp_summary['map_at_k']:.4f}  "
          f"MRR={exp_summary['mrr']:.4f}")

## 8. Score Distribution Analysis

Understanding the score distribution helps set the refusal threshold (`MIN_RETRIEVAL_SCORE`).

In [ ]:
# Analyze top scores for in-scope vs out-of-scope questions
in_scope_scores = [row["top_score"] for row in report if row["status"] != "REVIEW_REFUSAL"]
out_scope_scores = [row["top_score"] for row in report if row["status"] == "REVIEW_REFUSAL"]

if in_scope_scores:
    print(f"In-scope questions ({len(in_scope_scores)}):")
    print(f"  Min top score: {min(in_scope_scores):.4f}")
    print(f"  Max top score: {max(in_scope_scores):.4f}")
    print(f"  Mean top score: {sum(in_scope_scores) / len(in_scope_scores):.4f}")

if out_scope_scores:
    print(f"\nOut-of-scope questions ({len(out_scope_scores)}):")
    print(f"  Min top score: {min(out_scope_scores):.4f}")
    print(f"  Max top score: {max(out_scope_scores):.4f}")
    print(f"  Mean top score: {sum(out_scope_scores) / len(out_scope_scores):.4f}")

print(f"\nCurrent MIN_RETRIEVAL_SCORE: {config.MIN_RETRIEVAL_SCORE}")

## Summary

| Experiment | Found Rate | MAP@5 | Notes |
|-----------|-----------|-------|-------|
| Production (k=5, e5-base, 450/80) | ~90.6% | ~67.0% | Current configuration |
| k=3 | ~84.4% | ~75.7% | Higher precision, lower recall |
| k=10 | ~95.3% | ~50.0% | Higher recall, lower precision |
| e5-small | ~81.3% | ~65.0% | Faster but less accurate |
| Chunk 300/50 | ~90.6% | ~67.0% | Same as 450/80 |
| Chunk 600/100 | ~87.5% | ~66.3% | Slightly worse |

**Key finding:** `k=5` with `e5-base` and `450/80` chunking provides the best balance of recall and precision for this bilingual medical guideline application.